# PubMedCLIP Backbone Verification

**Model**: `flaviagiammarino/pubmed-clip-vit-base-patch32`  
**Arch**: CLIP ViT-B/32 fine-tuned on PubMed figure-caption pairs  
**Embed dim**: 512 (CLIP joint image-text space, L2-normalized)  
**Input**: 3×224×224 px, standard CLIP normalization  
**Access**: HuggingFace, no auth required  
**Zero-shot**: Yes — paired CLIP text encoder

Paper: [Does CLIP Benefit Visual Question Answering in the Medical Domain as Much as it Does in the General Domain? (arXiv 2112.13906)](https://arxiv.org/abs/2112.13906)

Note: PubMedCLIP was trained on figure-caption pairs from PubMed articles (not CXR-specific).
It covers diverse medical imaging modalities. For CXR-specific zero-shot, BiomedCLIP
or CheXzero are usually stronger.

In [ ]:
%pip install -q transformers torch torchvision Pillow

In [ ]:
import os

# Control which GPU(s) are visible to this notebook.
# Set CUDA_VISIBLE_DEVICES in your shell before launching Jupyter to override,
# e.g.:  CUDA_VISIBLE_DEVICES=2 jupyter lab
# Or change the default value below.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
print(f"CUDA_VISIBLE_DEVICES = {os.environ['CUDA_VISIBLE_DEVICES']}")

## 1. Load model via transformers CLIPModel

In [ ]:
import torch
from transformers import CLIPModel, CLIPProcessor

HUB = "flaviagiammarino/pubmed-clip-vit-base-patch32"

model     = CLIPModel.from_pretrained(HUB)
processor = CLIPProcessor.from_pretrained(HUB)
model.eval()

print(type(model))                          # CLIPModel
print(type(model.vision_model))             # CLIPVisionModel (ViT-B/32)
print("vision hidden size:",
      model.config.vision_config.hidden_size)   # 768 (ViT-B/32 internal dim)
print("projection dim:",
      model.config.projection_dim)          # 512 (joint space)

## 2. Image transform via CLIPProcessor

Use `CLIPProcessor` for preprocessing — it handles resize, center-crop, and the
CLIP-specific normalization (`mean=[0.48145466, 0.4578275, 0.40821073]`,
`std=[0.26862954, 0.26130258, 0.27577711]`).

In [ ]:
from PIL import Image
import numpy as np

def preprocess(path: str) -> dict:
    """Load CXR and return processor output (pixel_values tensor)."""
    img = Image.open(path).convert("RGB")
    return processor(images=img, return_tensors="pt")

# Show what the processor does
print("Processor image mean:", processor.image_processor.image_mean)
print("Processor image std: ", processor.image_processor.image_std)
print("Processor crop size: ", processor.image_processor.crop_size)

# Dummy batch
dummy_pil = Image.fromarray(np.ones((512, 512, 3), dtype=np.uint8) * 128)
inputs = processor(images=dummy_pil, return_tensors="pt")
print("pixel_values shape:", inputs["pixel_values"].shape)   # [1, 3, 224, 224]

## 3. Extract image embeddings

Three equivalent ways to get the joint-space image embedding:

In [ ]:
import torch.nn.functional as F

pv = inputs["pixel_values"]

with torch.no_grad():
    # Option A — convenience method (recommended)
    img_emb_A = model.get_image_features(pixel_values=pv)  # [B, 512], NOT normalized

    # Option B — full model forward
    out       = model(pixel_values=pv, input_ids=None,
                      attention_mask=None, return_loss=False)
    img_emb_B = out.image_embeds   # [B, 512], NOT normalized

    # Option C — vision tower only (768-dim, before projection)
    vis_out   = model.vision_model(pixel_values=pv)
    raw_cls   = vis_out.pooler_output   # [B, 768] — ViT CLS pooled

# L2-normalize for cosine similarity
img_emb_norm = F.normalize(img_emb_A, dim=-1)   # [B, 512]

print("joint-space embedding (A):", img_emb_A.shape)   # [1, 512]
print("joint-space embedding (B):", img_emb_B.shape)   # [1, 512]
print("raw vision CLS (C):       ", raw_cls.shape)     # [1, 768]
print("A ≈ B:", torch.allclose(img_emb_A, img_emb_B, atol=1e-4))

## Segmentation mode — patch tokens

For dense prediction, extract the per-patch token sequence from `vision_model.last_hidden_state`.
PubMedCLIP uses **ViT-B/32** with 32-px patches — the 224×224 input yields only a
**7×7** (49-token) patch grid, which is coarse for segmentation. Consider a ViT-B/16 or
ViT-L/14 backbone for finer spatial resolution.

In [ ]:
import math

# Extract patch tokens from the CLIP vision tower's last hidden state.
# ViT-B/32 at 224px: 7×7 = 49 patches (32-px patches), D_vision=768 (pre-projection).
with torch.no_grad():
    vis = model.vision_model(pixel_values=pv)

patches = vis.last_hidden_state[:, 1:]   # [B, N_patches, D_vision] — skip CLS

B, N, D = patches.shape
H = W = int(math.sqrt(N))   # 7 for 224px / 32px-patches (ViT-B/32)
assert H * W == N

spatial = patches.permute(0, 2, 1).reshape(B, D, H, W)
print(f"patch tokens:  {patches.shape}")   # [1, 49, 768]
print(f"spatial map:   {spatial.shape}")   # [1, 768, 7, 7]

## 4. Zero-shot: text embeddings & similarity

In [ ]:
prompts = [
    "A chest X-ray showing pleural effusion.",
    "A normal chest radiograph with no acute findings.",
    "Cardiomegaly on chest X-ray.",
    "Pneumonia infiltrate visible.",
]

text_inputs = processor(text=prompts, return_tensors="pt",
                        padding=True, truncation=True)

with torch.no_grad():
    text_emb = model.get_text_features(**text_inputs)   # [4, 512]
    text_emb = F.normalize(text_emb, dim=-1)

sim = (img_emb_norm @ text_emb.T).squeeze(0)   # [4]
for p, s in zip(prompts, sim.tolist()):
    print(f"{s:.4f}  {p}")

## Notes for RadHarmony integration

```
model_call : lambda m, x: F.normalize(m.get_image_features(pixel_values=x), dim=-1)
pool       : None
embed_dim  : 512  (joint CLIP space)
             768  (if using raw vision CLS: model.vision_model(pixel_values=x).pooler_output)
transform  : CLIPProcessor — Resize(224) → CenterCrop(224) → ToTensor
             → Normalize(CLIP mean/std)
```

The `CLIPProcessor` outputs `pixel_values` — in the MONAI/RadHarmony pipeline, use
`EncoderPreprocessTransform.from_huggingface(keys=["img"], processor=processor)`
as the transform, which handles the processor call and returns a tensor.

**Vs BiomedCLIP**: PubMedCLIP is trained on mixed-modality PubMed figures; BiomedCLIP
is trained on 15M figure-caption pairs with domain-specific adaptations. For CXR tasks
BiomedCLIP typically outperforms PubMedCLIP.

**Segmentation mode** (`output_keys={"img", "mask"}`):
- Patch tokens: `model.vision_model(pixel_values=x).last_hidden_state[:, 1:]` → `[B, 49, 768]`
- Spatial map: reshape to `[B, 768, 7, 7]` (224px / 32px-patches → 7×7)
- ⚠️ 7×7 is a **coarse** grid — prefer a ViT-B/16 backbone for finer segmentation resolution.

In [ ]:
CXR_PATH = "/path/to/cxr.jpg"

img = Image.open(CXR_PATH).convert("RGB")
inputs = processor(images=img, return_tensors="pt")

with torch.no_grad():
    feats = F.normalize(model.get_image_features(**inputs), dim=-1)  # [1, 512]

sim = (feats @ text_emb.T).squeeze(0)
for p, s in sorted(zip(prompts, sim.tolist()), key=lambda x: -x[1]):
    print(f"{s:.4f}  {p}")

## Notes for RadHarmony integration

```
model_call : lambda m, x: F.normalize(m.get_image_features(pixel_values=x), dim=-1)
pool       : None
embed_dim  : 512  (joint CLIP space)
             768  (if using raw vision CLS: model.vision_model(pixel_values=x).pooler_output)
transform  : CLIPProcessor — Resize(224) → CenterCrop(224) → ToTensor
             → Normalize(CLIP mean/std)
```

The `CLIPProcessor` outputs `pixel_values` — in the MONAI/RadHarmony pipeline, use
`EncoderPreprocessTransform.from_huggingface(keys=["img"], processor=processor)`
as the transform, which handles the processor call and returns a tensor.

**Vs BiomedCLIP**: PubMedCLIP is trained on mixed-modality PubMed figures; BiomedCLIP
is trained on 15M figure-caption pairs with domain-specific adaptations. For CXR tasks
BiomedCLIP typically outperforms PubMedCLIP.